# Assignment 5 - Evaluation metrics

RAGAS Metrics: https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/

## LLM As Judge
1. [Noise sensitivity](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/noise_sensitivity/) - % incorrect claims out of total claims
  - Response
  - Context
2. [Faithfulness](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/faithfulness/) - % of claims supported by retrieved context
  - Response
  - Context
3. Persona preferences - Does the response reflect the preferences of the persona? [0,1,2] scale
  - Response
  - Persona description
  - Gold response example (multi-shot)
  - report as normalized score [0,1]
4. [Answer relevancy](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/answer_relevance/#answer-relevancy) - relevancy of the Gold question to the generated response
  - Gold question
  - Generated response
  - Judge generates 3 questions based on the response
  - Average cosine similarity of generated questions to the gold question


## Mathematical Eval
5. [BERT Score](https://arxiv.org/abs/1904.09675) - compares embeddings of gold asnwer to generates response
  - Gold answer
  - Generated response
6. [Honesty/Hallucination](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/agents/#topic-adherence) - Does the LLM return the "I don't know" phrase [IDK] when it cannot find relevant context (F1 score).
  - Precision = (IDK & NO Context) / (IDK & Have Context + IDK & NO Context)
  - Recall = (IDK & NO Context) / (Factual response & NO Context + IDK Response & NO Context)
  - F1 Score = 2*Precision*Recall / (Precision + Recall)


===========================================================================================================

## 1. Setup

We will first install a number of libraries and import what we will need.





In [2]:
from google.colab import userdata

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%%capture
!pip install -q -U langchain langchain-community langchain-huggingface langchain-qdrant
!pip install -q -U qdrant-client sentence-transformers arxiv pymupdf
!pip install -q -U transformers accelerate bitsandbytes

!pip install -q -U xmltodict

!pip install -q -U cohere

!pip install -q -U langchain-cohere

!pip install -q -U wikipedia


In [4]:
import os
import numpy as np
import time
import locale

# IMPORTANT: Add your Hugging Face token to Colab's Secrets (the key icon on the left panel)
# and name it 'HF_TOKEN', or replace the line below with os.environ["HF_TOKEN"] = "your_token"
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

COHERE_API_KEY = userdata.get('COHERE_API_KEY')

import langchain
from langchain_community.document_loaders import ArxivLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_qdrant import QdrantVectorStore

from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.utils.function_calling import convert_to_openai_tool


from langchain_community.document_loaders import ArxivLoader
from langchain_community.document_loaders import PyMuPDFLoader

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig


from langchain_cohere import ChatCohere

import torch.nn.functional as F
from torch import Tensor
from transformers import AutoModel

import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import WikipediaLoader

locale.getpreferredencoding = lambda: "UTF-8"


/tmp/ipykernel_501/1138715678.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import ArxivLoader


In [5]:
from typing import Annotated, Literal, TypedDict
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from transformers.utils import get_json_schema
from typing import Annotated

In [ ]:
%%capture
!pip install -U sentence_transformers
from sentence_transformers import CrossEncoder


!tar -xzf "/content/drive/MyDrive/Colab Data/MIDS-267-A5/qdrant_250_50.tar.gz" -C /content

In [ ]:
%%capture

class VectorStoreRetriever():
      EMBEDDINGS_MODEL = 'sentence-transformers/multi-qa-mpnet-base-dot-v1'
      def __init__(self):
          self.collection_name = c
          path = '/content/qdrant_storage'
          self.init_embeddings(self.EMBEDDINGS_MODEL)
          self.init_vector_store(path, self.collection_name)
      def init_vector_store(self, path, collection_name):
          self.vector_store = QdrantVectorStore(
              client=QdrantClient(path=path),
              embedding=self.base_embeddings,
              collection_name=collection_name,
              distance=Distance.DOT)
      def init_embeddings(self, embeddings_model):
          self.base_embeddings = HuggingFaceEmbeddings(model_name=embeddings_model)


vector_store = VectorStoreRetriever().vector_store

In [ ]:
%%capture

class CrossEncoderWrapper():
    def __init__(self, threshold, doc_limit):
        self.threshold = threshold
        self.doc_limit = doc_limit
        self.cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", activation_fn=torch.nn.Sigmoid())
    def scores(self, query, context):
        return self.cross_encoder.predict([(query, doc.page_content) for doc in context])
    def filtered(self, query, context):
        scores = self.scores(query, context)
        reranked_docs = [(score, d) for score, d in sorted(zip(scores, context), key=lambda x: x[0], reverse=True)]
        return list(filter(lambda reranked: reranked[0] >= self.threshold, reranked_docs))[:self.doc_limit]

cross_encoder = CrossEncoderWrapper(threshold=0.5, doc_limit=5)


In [6]:
# LOAD QWEN 3 LLM
# judge for Gold Answer generation
qwen_model_name = "Qwen/Qwen3-8B"

qwen_quantization_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.bfloat16
)

# load the tokenizer and the model
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    dtype=torch.float32,
    device_map="auto",
    max_length = None,
    quantization_config=qwen_quantization_config
)
qwen_model.config.pad_token_id = qwen_model.config.eos_token_id

qwen_pipe = pipeline(
    "text-generation",
    model=qwen_model,
    tokenizer=qwen_tokenizer,
    max_new_tokens=1000,
    temperature=0.3,
    top_p=0.5,
    do_sample=True,
    repetition_penalty=1.2
)

qwen_llm = HuggingFacePipeline(pipeline=qwen_pipe)
qwen_chat = ChatHuggingFace(llm=qwen_llm)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'repetition_penalty', 'top_p', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


### Noise sensitivity
https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/noise_sensitivity/

Definition: % incorrect claims out of total claims

Required data:
  - Response
  - Context

## Faithfulness
https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/faithfulness/

Definition: % of claims supported by retrieved context

- Response
- Context

In [ ]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.prompts import PromptTemplate
import re
import json

def extract_and_parse_json(text_output: str):
    print(text_output)
    if "</think>" in text_output:
        text_output = text_output.split("</think>")[1]
    try:
        return json.loads(text_otput)
    except json.JSONDecodeError as e:
        raise ValueError(f"Failed to parse extracted JSON: {e}\nJSON string: {json_str}") from e

claims_template = """Your task is to evaluate the factual claims of a provided statement based ONLY on provided context.

Factual claims are classified in 3 possible categories: noisy, faithful, or irrelevant.
A claim is faithful if it is supported by the supplied context. The factual basis of a faituful claim MUST appear in the context.
A claim is noisy if it is incorrect, facturally wrong, or contrary to the information in the context.
Claims that are not noisy and not faithful are irrelevant - the context does not mention the facts of the claim, so it cannot be determined as noisy or faithful to the context

Context:
{context}

Statement:
{statement}

---
CRITICAL INSTRUCTION: Your response MUST be a valid JSON object, and ONLY a JSON object, wrapped in a markdown code block. No other text, conversation, or explanation is permitted. The JSON block must be the *entirety* of your response.
---

```json
{{
"claims": strings[array of factual claims],
"claims_count": integer (total count of all claims),
"noisy_count": integer (count of noisy claims),
"faithful_count": integer (count of faithful claims),
"irrelevant_count": integer (count of irrelevant claims)
}}
```"""


claims_prompt_template = PromptTemplate(template=claims_template,
                                        input_variables=["context", "statement"])

claims_chain = (
    claims_prompt_template
    | qwen_chat
    | StrOutputParser()
    | extract_and_parse_json
)

print(claims_chain.invoke({"context": "Frogs are green.", "statement": "Darryl is a frog, therefore he is green."}))


[transformers] Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
import random

def get_random_chunks(count=1):

    collection = "rag_tech_db_250"
    collection_info = vector_store.client.get_collection(collection_name=collection)
    total_points = collection_info.points_count

    records, _ = vector_store.client.scroll(
        collection_name=collection,
        limit=total_points,
        with_payload=False,
    )
    all_ids = [record.id for record in records]

    random_ids = random.sample(all_ids, count)

    random_chunks = vector_store.client.retrieve(
        collection_name=collection,
        ids=random_ids,
        with_payload=True,
    )

    return random_chunks


In [ ]:
import json

test_questions = {}

for n, chunk in enumerate(get_random_chunks(count=150)):
    n += 1
    context = chunk.payload['page_content']
    text = llm_prompt_template.format(context=context)
    prompt = qwen_tokenizer.apply_chat_template(
        [{"role": "user", "content": text}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    question = qwen_pipe(prompt, return_full_text=False)[0]["generated_text"]

    out = {
        "question": question,
        'question_id': n,
        'context_id': chunk.id,
        'metadata': chunk.payload['metadata']
    }
    test_questions[n] = out

    print(f"Question {n}: {question}")
    print(out)
test_q_path = f"/content/drive/MyDrive/Colab Data/MIDS-267-A5/test_questions_qwen.json"
json.dump(test_questions, open(test_q_path, "w"), indent=2)

Question 1: What does the research suggest about the relationship between model capabilities, training costs, and inference costs in the context of language models?
{'question': 'What does the research suggest about the relationship between model capabilities, training costs, and inference costs in the context of language models?', 'question_id': 1, 'context_id': '9ed4cc8ffc5f474dbb2b8de949c84cbc', 'metadata': {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-10-11T00:48:17+00:00', 'source': 'https://arxiv.org/pdf/2310.06825.pdf', 'file_path': 'https://arxiv.org/pdf/2310.06825.pdf', 'total_pages': 9, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-10-11T00:48:17+00:00', 'trapped': '', 'modDate': 'D:20231011004817Z', 'creationDate': 'D:20231011004817Z', 'page': 5, 'page_num': 5, 'doc_source': 'ArXiv', 'id': '2310.06825', 'split_id': 21, 'doc_num': 20}}
Question 2: What recent research explores methods t

In [ ]:
# clean the questions of self references and meta references
test_qs = json.loads(Path(test_q_path).read_text())

BANNED = ("title", "paper", "examined", "et al.", "最佳", "[skip]",
          "described", "mathbf$", "mentioned", "provided", "the given references",
          "the given context", "the research suggest",
          "according to the given")

test_qs = {k: v for k, v in test_qs.items() if  not any(b in v['question'].lower() for b in BANNED)}

json.dump(test_qs, open("/content/drive/MyDrive/Colab Data/MIDS-267-A5/test_questions_qwen_cleaned.json", "w"), indent=2)